# Gold Direction ML — Research-Grade Pipeline

A meta-labeled, volatility-aware machine-learning pipeline for trading gold.

This notebook upgrades a basic fixed-horizon XGBoost classifier into a rigorous
research workflow:

1. **Expanded features** — momentum, volatility/trend regimes, cross-asset,
   calendar, and a rolling **ARIMA forecast** as an input feature.
2. **Triple-barrier + meta-labeling** (López de Prado) — volatility-scaled,
   path-aware labels with a principled *abstain* mechanism.
3. **Purged walk-forward cross-validation** with Optuna hyperparameter search,
   optimized on a financial objective (out-of-fold strategy Sharpe).
4. **Probability calibration + decision-threshold tuning** for risk-adjusted return.
5. **Model comparison** (XGBoost vs LightGBM vs logistic vs **LSTM**) **+ SHAP**
   explainability.
6. **Event-driven, cost-aware backtest** with Sharpe / Sortino / max-drawdown /
   per-regime breakdown.

> Requires network access (yfinance) to execute.

In [ ]:
# Install extra dependencies (safe to re-run; quiet)
# tensorflow-cpu powers the LSTM baseline; statsmodels powers the ARIMA feature.
%pip install -q optuna shap lightgbm statsmodels tensorflow-cpu

## 1. Setup & Data

`fetch_macro_data` is unchanged from the baseline: forward-fill only (no
back-fill lookahead), with MultiIndex handling. We add the S&P 500 (`^GSPC`) to
support cross-asset / risk-on-risk-off features.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

def fetch_macro_data(tickers_dict, start_date="2000-01-01", end_date="2026-01-01"):
    tickers_list = list(tickers_dict.keys())
    print(f"Fetching macro data for {tickers_list}...")
    raw_df = yf.download(tickers_list, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)
    if isinstance(raw_df.columns, pd.MultiIndex):
        close_df = raw_df["Close"].copy()
    else:
        close_df = raw_df[["Close"]].copy()
    close_df = close_df.rename(columns=tickers_dict)
    # Forward-fill only: back-filling leading NaNs would leak future values into
    # the warm-up period. Warm-up rows are dropped later via dropna().
    return close_df.ffill()

CONFIG = {
    # technical windows
    "rsi_window": 14, "rsi_window_short": 5,
    "macd_fast": 12, "macd_slow": 26, "macd_signal": 9,
    "sma_short": 50, "sma_long": 200,
    "vol_window": 21, "vol_window_short": 10,
    "ratio_z_window": 252, "vix_z_window": 252, "corr_window": 60,
    # triple-barrier labeling
    "pt_mult": 2.0, "sl_mult": 2.0, "max_holding_days": 10,
    # ARIMA forecast feature (leak-free rolling refit)
    "arima_order": (1, 0, 1), "arima_window": 252,
    "arima_horizon": 10, "arima_refit_every": 21,
    # LSTM comparison baseline
    "lstm_seq_len": 20, "lstm_epochs": 15,
    # walk-forward CV + search  (keep small for fast runs; raise for production)
    "n_splits": 5, "embargo": 5, "n_trials": 30, "holdout_frac": 0.20,
    # backtest
    "cost_per_trade": 0.0005, "trading_days": 252,
    "random_state": 42,
}

tickers_map = {
    "GC=F": "Gold_Close",
    "DX-Y.NYB": "DXY_Close",
    "^VIX": "VIX_Close",
    "^TNX": "TNX_Close",
    "SI=F": "Silver_Close",
    "^GSPC": "SPX_Close",
}

raw_data = fetch_macro_data(tickers_map)
print("Data fetched successfully.", raw_data.shape)

## 2. Feature Engineering

All features use only past data (`rolling` / `ewm` / `shift`). The engineered
feature names are centralized in `FEATURES` so the model, SHAP, and the backtest
read from a single source of truth. A couple of columns (`High_Vol_Regime`,
`Trend_Sign`) are also kept for the per-regime backtest breakdown.

In [ ]:
def engineer_features(df, config=CONFIG):
    print("Engineering features...")
    p = df.copy().sort_index()
    c = p["Gold_Close"]
    log_ret = np.log(c / c.shift(1))

    # --- RSI (two windows) ---
    def rsi(series, window):
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        com = window - 1
        eg = gain.ewm(com=com, adjust=False).mean()
        el = loss.ewm(com=com, adjust=False).mean()
        return 100 - (100 / (1 + eg / (el + 1e-10)))
    p["Gold_RSI"] = rsi(c, config["rsi_window"])
    p["Gold_RSI_5"] = rsi(c, config["rsi_window_short"])

    # --- Normalized MACD histogram (scaled by price) ---
    exp1 = c.ewm(span=config["macd_fast"], adjust=False).mean()
    exp2 = c.ewm(span=config["macd_slow"], adjust=False).mean()
    macd_line = (exp1 - exp2) / c
    signal_line = macd_line.ewm(span=config["macd_signal"], adjust=False).mean()
    p["Gold_MACD_Hist"] = macd_line - signal_line

    # --- Trend ---
    sma_s = c.rolling(config["sma_short"]).mean()
    sma_l = c.rolling(config["sma_long"]).mean()
    p["Dist_SMA_50"] = (c - sma_s) / sma_s
    p["Dist_SMA_200"] = (c - sma_l) / sma_l
    p["SMA_Slope_200"] = sma_l.pct_change(periods=21)
    p["Trend_Sign"] = (sma_s > sma_l).astype(int)          # golden/death cross

    # --- Momentum (lagged log returns) ---
    for k in (1, 5, 10, 21):
        p[f"Mom_{k}"] = np.log(c / c.shift(k))

    # --- Volatility & regime ---
    p["Gold_Vol"] = log_ret.rolling(config["vol_window"]).std()
    vol_s = log_ret.rolling(config["vol_window_short"]).std()
    p["Vol_Ratio"] = vol_s / (p["Gold_Vol"] + 1e-10)
    vol_med = p["Gold_Vol"].rolling(config["ratio_z_window"]).median()
    p["High_Vol_Regime"] = (p["Gold_Vol"] > vol_med).astype(int)

    # --- Macro ---
    if "DXY_Close" in p:
        p["DXY_Pct_Change"] = p["DXY_Close"].pct_change()
    if "TNX_Close" in p:
        p["TNX_Diff"] = p["TNX_Close"].diff() / 100.0
    if "VIX_Close" in p:
        p["VIX_3d_Diff"] = p["VIX_Close"].diff(periods=3)
        vmean = p["VIX_Close"].rolling(config["vix_z_window"]).mean()
        vstd = p["VIX_Close"].rolling(config["vix_z_window"]).std()
        p["VIX_Z"] = (p["VIX_Close"] - vmean) / (vstd + 1e-10)
    if "Silver_Close" in p:
        # rolling z-score of the gold/silver ratio (stationary; trees can use it)
        ratio = c / p["Silver_Close"]
        roll = ratio.rolling(config["ratio_z_window"])
        p["Gold_Silver_Ratio_Z"] = (ratio - roll.mean()) / (roll.std() + 1e-10)
        sil_mom = np.log(p["Silver_Close"] / p["Silver_Close"].shift(21))
        p["Gold_Silver_Mom_Spread"] = p["Mom_21"] - sil_mom

    # --- Cross-asset ---
    w = config["corr_window"]
    if "SPX_Close" in p:
        spx_mom = np.log(p["SPX_Close"] / p["SPX_Close"].shift(21))
        p["Gold_SPX_Mom_Spread"] = p["Mom_21"] - spx_mom
    if "DXY_Close" in p:
        p["Corr_Gold_DXY"] = log_ret.rolling(w).corr(
            np.log(p["DXY_Close"] / p["DXY_Close"].shift(1)))
    if "TNX_Close" in p:
        p["Corr_Gold_TNX"] = log_ret.rolling(w).corr(p["TNX_Close"].diff())

    # --- Calendar ---
    p["DayOfWeek"] = p.index.dayofweek
    p["Month"] = p.index.month
    p["TurnOfMonth"] = ((p.index.day <= 3) | (p.index.day >= 26)).astype(int)

    return p

FEATURES = [
    "Gold_RSI", "Gold_RSI_5", "Gold_MACD_Hist",
    "Dist_SMA_50", "Dist_SMA_200", "SMA_Slope_200", "Trend_Sign",
    "Mom_1", "Mom_5", "Mom_10", "Mom_21",
    "Gold_Vol", "Vol_Ratio", "High_Vol_Regime",
    "DXY_Pct_Change", "TNX_Diff", "VIX_3d_Diff", "VIX_Z",
    "Gold_Silver_Ratio_Z", "Gold_Silver_Mom_Spread",
    "Gold_SPX_Mom_Spread", "Corr_Gold_DXY", "Corr_Gold_TNX",
    "DayOfWeek", "Month", "TurnOfMonth",
]

featured = engineer_features(raw_data)
print(f"Engineered {len(FEATURES)} features.")

### 2b. ARIMA forecast as a feature

Rather than use ARIMA as a standalone predictor (daily gold returns are near
white noise, so its forecasts have little direct edge), we fold it into the
model as a **single feature**: a rolling ARIMA fit on trailing gold returns
produces an `h`-step cumulative expected-return forecast. The model is refit
every `arima_refit_every` days on a trailing window, and forecasts are
forward-filled — so the value at any date uses only past returns (leak-free).

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

def compute_arima_feature(close, config=CONFIG):
    """Rolling ARIMA h-step cumulative expected-return forecast (leak-free)."""
    ret = np.log(close / close.shift(1)).dropna()
    idx = ret.index
    order = config["arima_order"]
    win, h, step = config["arima_window"], config["arima_horizon"], config["arima_refit_every"]
    fc = pd.Series(index=close.index, dtype=float)
    for i in range(win, len(ret), step):
        window = ret.iloc[i - win:i]            # returns through idx[i-1]
        try:
            res = ARIMA(window, order=order).fit()
            fc.loc[idx[i]] = float(res.forecast(steps=h).sum())
        except Exception:
            fc.loc[idx[i]] = np.nan
    return fc.ffill()                           # carry last past forecast forward

print("Computing rolling ARIMA forecast feature (this can take a minute)...")
featured["ARIMA_Forecast"] = compute_arima_feature(featured["Gold_Close"], CONFIG)
if "ARIMA_Forecast" not in FEATURES:
    FEATURES.append("ARIMA_Forecast")
print(f"Feature set now has {len(FEATURES)} features (added ARIMA_Forecast).")

## 3. Triple-Barrier Labeling + Meta-Labeling

Instead of a fixed-horizon up/down label, we use **triple-barrier labeling**:

- **Primary signal** — a simple long-only trend filter (`Gold_Close > SMA50`)
  generates the candidate trades (events).
- **Three barriers** per event: a volatility-scaled profit-take (`+pt_mult·σ`),
  a stop-loss (`−sl_mult·σ`), and a vertical/time barrier at `max_holding_days`.
  The label is decided by **whichever barrier is touched first**, and we record
  the realized return and holding period.
- **Meta-label** — the ML model predicts *whether the primary signal will be
  profitable* (1 if the realized return is positive). The model thus decides
  **whether to take** each signal; the tuned threshold controls how selective.
- **Sample weights** — overlapping events are down-weighted by their *average
  uniqueness* so concurrent labels don't over-count.

In [ ]:
def get_daily_vol(close, span):
    """EWMA standard deviation of daily log returns (fractional)."""
    return np.log(close / close.shift(1)).ewm(span=span).std()

def apply_triple_barrier(close, events_idx, vol, config=CONFIG):
    """For each event entry, return the first-touched barrier's outcome.

    Barriers are volatility-scaled: +pt_mult*sigma / -sl_mult*sigma, with a
    vertical barrier at max_holding_days. Returns label/ret/t1/holding.
    """
    vals = close.values
    idx = close.index
    pt, sl, max_h = config["pt_mult"], config["sl_mult"], config["max_holding_days"]
    rows = []
    for t0 in events_idx:
        i0 = idx.get_loc(t0)
        v = vol.loc[t0]
        if np.isnan(v) or v == 0:
            continue
        entry = vals[i0]
        up, dn = entry * (1 + pt * v), entry * (1 - sl * v)
        i_end = min(i0 + max_h, len(vals) - 1)
        exit_i = i_end
        for j in range(i0 + 1, i_end + 1):
            if vals[j] >= up or vals[j] <= dn:
                exit_i = j
                break
        ret = vals[exit_i] / entry - 1.0
        rows.append((t0, idx[exit_i], ret, exit_i - i0))
    out = pd.DataFrame(rows, columns=["t0", "t1", "ret", "holding"]).set_index("t0")
    out["label"] = (out["ret"] > 0).astype(int)   # meta-label
    return out

def average_uniqueness_weights(events, bar_index):
    """Average-uniqueness sample weights (de Prado): down-weight overlap."""
    i0 = bar_index.get_indexer(events.index)
    i1 = bar_index.get_indexer(events["t1"].values)
    count = np.zeros(len(bar_index))
    for a, b in zip(i0, i1):
        count[a:b + 1] += 1.0
    w = np.array([np.mean(1.0 / count[a:b + 1]) for a, b in zip(i0, i1)])
    return w / w.mean()                            # normalize to mean 1

# --- Build events from the primary signal, then label them ---
close = featured["Gold_Close"]
daily_vol = get_daily_vol(close, CONFIG["vol_window"])
sma_s = close.rolling(CONFIG["sma_short"]).mean()
primary_signal = close > sma_s                     # long-only trend filter

# valid feature rows only, signal active
valid = featured[FEATURES].notna().all(axis=1) & primary_signal & daily_vol.notna()
events_idx = featured.index[valid]

events = apply_triple_barrier(close, events_idx, daily_vol, CONFIG)
events["weight"] = average_uniqueness_weights(events, featured.index)
print(f"Generated {len(events)} labeled events. "
      f"Positive rate: {events['label'].mean():.3f}")

## 4. Assemble Dataset & Time-Ordered Holdout

We align features to event entry times, then carve off the most recent
`holdout_frac` of events as a final out-of-sample set that the hyperparameter
search never sees.

In [ ]:
X_all = featured.loc[events.index, FEATURES].copy()
y_all = events["label"].copy()
w_all = events["weight"].values
ret_all = events["ret"].copy()
t1_all = events["t1"].copy()

# label-end position within the EVENTS array (for purging in CV)
event_times = X_all.index.values
t1_pos_all = np.searchsorted(event_times, events["t1"].values)

# time-ordered split
n = len(X_all)
split = int(n * (1 - CONFIG["holdout_frac"]))
X_tr, X_ho = X_all.iloc[:split], X_all.iloc[split:]
y_tr, y_ho = y_all.iloc[:split], y_all.iloc[split:]
w_tr = w_all[:split]
ret_tr, ret_ho = ret_all.iloc[:split], ret_all.iloc[split:]
t1pos_tr = np.clip(t1_pos_all[:split], 0, split - 1)

assert not X_all.isna().any().any(), "NaNs leaked into the feature matrix"
print(f"Train events: {len(X_tr)}  |  Holdout events: {len(X_ho)}")
print(f"Train positive rate: {y_tr.mean():.3f}  |  Holdout: {y_ho.mean():.3f}")

## 5. Purged Walk-Forward Cross-Validation

Standard k-fold leaks across time. We use an **expanding walk-forward** split
where, for each test fold, training samples whose label window overlaps the test
period (plus an `embargo`) are **purged** — eliminating look-ahead from
overlapping triple-barrier labels.

In [ ]:
class PurgedWalkForwardCV:
    """Expanding walk-forward splits with purging + embargo (positional)."""
    def __init__(self, n_splits, t1_pos, embargo=0):
        self.n_splits = n_splits
        self.t1_pos = np.asarray(t1_pos)
        self.embargo = embargo

    def split(self, X):
        n = len(X)
        indices = np.arange(n)
        # test folds = blocks 1..n_splits (block 0 is the initial training seed)
        test_folds = np.array_split(indices, self.n_splits + 1)[1:]
        for test_idx in test_folds:
            test_start = test_idx[0]
            train_idx = indices[:test_start]
            # purge train samples whose label extends to/after (test_start-embargo)
            keep = self.t1_pos[train_idx] < (test_start - self.embargo)
            train_idx = train_idx[keep]
            if len(train_idx) == 0:
                continue
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

cv = PurgedWalkForwardCV(CONFIG["n_splits"], t1pos_tr, CONFIG["embargo"])

# sanity check: no train label-end bleeds past the embargo into any test fold
for tr, te in cv.split(X_tr):
    assert t1pos_tr[tr].max() < te[0] - CONFIG["embargo"] + 1
print(f"{cv.get_n_splits()} purged walk-forward folds constructed.")

## 6. Hyperparameter Search (Optuna, financial objective)

The objective is the **mean out-of-fold strategy Sharpe** across walk-forward
folds — not raw accuracy — so tuning optimizes what we care about: risk-adjusted
trading performance. Sample weights are passed to every fit.

In [ ]:
import optuna
from xgboost import XGBClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)

def fold_sharpe(proba, ret_te, threshold=0.5):
    sig = (proba >= threshold).astype(int)
    if sig.sum() == 0:
        return 0.0
    r = sig * ret_te
    return float(r.mean() / (r.std() + 1e-9))

def cv_score(params, X, y, w, ret, cv):
    sharpes = []
    for tr, te in cv.split(X):
        m = XGBClassifier(**params, eval_metric="logloss",
                          random_state=CONFIG["random_state"], n_jobs=-1)
        m.fit(X.iloc[tr], y.iloc[tr], sample_weight=w[tr])
        p = m.predict_proba(X.iloc[te])[:, 1]
        sharpes.append(fold_sharpe(p, ret.iloc[te].values))
    return float(np.mean(sharpes))

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    }
    return cv_score(params, X_tr, y_tr, w_tr, ret_tr, cv)

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=CONFIG["random_state"]))
study.optimize(objective, n_trials=CONFIG["n_trials"], show_progress_bar=False)

best_params = study.best_params
print(f"Best out-of-fold strategy Sharpe: {study.best_value:.4f}")
print("Best params:", best_params)

## 7. Calibration & Threshold Tuning

We collect **out-of-fold** predictions on the training data with the best params
and tune the decision threshold to maximize OOF strategy Sharpe (leak-free). We
then fit the final model on all training data and wrap it in
`CalibratedClassifierCV` so the probabilities are well-calibrated.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

# --- out-of-fold predictions on train (best params) for threshold tuning ---
oof = np.full(len(y_tr), np.nan)
for tr, te in cv.split(X_tr):
    m = XGBClassifier(**best_params, eval_metric="logloss",
                      random_state=CONFIG["random_state"], n_jobs=-1)
    m.fit(X_tr.iloc[tr], y_tr.iloc[tr], sample_weight=w_tr[tr])
    oof[te] = m.predict_proba(X_tr.iloc[te])[:, 1]
oof_mask = ~np.isnan(oof)

best_t, best_s = 0.5, -np.inf
for t in np.linspace(0.30, 0.70, 41):
    sig = (oof[oof_mask] >= t).astype(int)
    if sig.sum() < 5:
        continue
    s = fold_sharpe(oof[oof_mask], ret_tr.values[oof_mask], threshold=t)
    if s > best_s:
        best_s, best_t = s, t
print(f"Tuned decision threshold: {best_t:.3f}  (OOF Sharpe {best_s:.4f})")

# --- final models fit on ALL training data ---
xgb_best = XGBClassifier(**best_params, eval_metric="logloss",
                         random_state=CONFIG["random_state"], n_jobs=-1)
xgb_best.fit(X_tr, y_tr, sample_weight=w_tr)            # plain model (for SHAP)

calibrated = CalibratedClassifierCV(
    XGBClassifier(**best_params, eval_metric="logloss",
                  random_state=CONFIG["random_state"], n_jobs=-1),
    method="isotonic", cv=3)
calibrated.fit(X_tr, y_tr, sample_weight=w_tr)         # calibrated probabilities

## 8. Out-of-Sample Evaluation

Metrics on the untouched holdout, using the tuned threshold. Accuracy is shown
against the majority-class baseline (the right reference, not 50%).

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix)
from sklearn.calibration import calibration_curve

proba_ho = calibrated.predict_proba(X_ho)[:, 1]
pred_ho = (proba_ho >= best_t).astype(int)

base_rate = y_ho.mean()
naive_acc = max(base_rate, 1 - base_rate)
print(f"Accuracy:           {accuracy_score(y_ho, pred_ho):.4f}")
print(f"Naive baseline acc: {naive_acc:.4f} (majority class)")
print(f"ROC-AUC:            {roc_auc_score(y_ho, proba_ho):.4f}")
print(f"Base rate (win):    {base_rate:.4f}\n")
print(classification_report(y_ho, pred_ho))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_ho, pred_ho)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[0],
            xticklabels=["No trade", "Take"], yticklabels=["Loss", "Win"])
ax[0].set_title("Confusion Matrix (holdout)", fontweight="bold")
ax[0].set_xlabel("Model decision"); ax[0].set_ylabel("Actual outcome")

frac_pos, mean_pred = calibration_curve(y_ho, proba_ho, n_bins=10, strategy="quantile")
ax[1].plot(mean_pred, frac_pos, "o-", label="Calibrated model")
ax[1].plot([0, 1], [0, 1], "--", color="gray", label="Perfect")
ax[1].set_title("Reliability Curve (holdout)", fontweight="bold")
ax[1].set_xlabel("Mean predicted probability"); ax[1].set_ylabel("Observed win rate")
ax[1].legend()
plt.tight_layout(); plt.show()

## 9. Model Comparison + SHAP

Benchmark the tuned XGBoost against LightGBM, a scaled logistic-regression
baseline, and an **LSTM sequence model** on the same holdout, then explain the
XGBoost model with SHAP.

The LSTM is a fair sequence-model benchmark: it ingests the last `lstm_seq_len`
days of (train-scaled) features ending at each event and predicts the meta-label.
Note its AUC/accuracy are computed on the events with enough lookback history
(the first few are trimmed), so its sample is marginally smaller than the
tabular models'. On low signal-to-noise tabular data, gradient-boosted trees
typically win — this quantifies whether sequence modeling adds anything here.

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import shap

def evaluate(name, est, fit_kwargs=None):
    fit_kwargs = fit_kwargs or {}
    est.fit(X_tr, y_tr, **fit_kwargs)
    p = est.predict_proba(X_ho)[:, 1]
    sig = (p >= 0.5).astype(int)
    r = sig * ret_ho.values
    sharpe = (r.mean() / (r.std() + 1e-9)) * np.sqrt(CONFIG["trading_days"]) if sig.sum() else np.nan
    return {"Model": name, "AUC": roc_auc_score(y_ho, p),
            "Accuracy": accuracy_score(y_ho, (p >= 0.5).astype(int)),
            "Strategy_Sharpe": sharpe, "Trades": int(sig.sum())}

rows = [
    evaluate("XGBoost (tuned)", xgb_best, {"sample_weight": w_tr}),
    evaluate("LightGBM", LGBMClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8,
        random_state=CONFIG["random_state"], n_jobs=-1, verbose=-1),
        {"sample_weight": w_tr}),
    evaluate("Logistic (scaled)", make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=0.5,
                           random_state=CONFIG["random_state"]))),
]

# --- LSTM sequence-model baseline (skipped gracefully if TF unavailable) ---
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    tf.random.set_seed(CONFIG["random_state"])

    seq_len = CONFIG["lstm_seq_len"]
    feat_all = featured[FEATURES].astype(float)
    # scale using ONLY data up to the train cutoff to avoid leakage
    sc = StandardScaler().fit(feat_all.loc[:X_tr.index[-1]].values)
    scaled = pd.DataFrame(sc.transform(feat_all.values),
                          index=feat_all.index, columns=FEATURES).fillna(0.0)
    arr = scaled.values
    pos = {t: i for i, t in enumerate(scaled.index)}

    def make_seqs(event_index):
        seqs, kept = [], []
        for t in event_index:
            i = pos[t]
            if i - seq_len + 1 >= 0:
                seqs.append(arr[i - seq_len + 1:i + 1])
                kept.append(t)
        return np.asarray(seqs), pd.Index(kept)

    Xs_tr, idx_tr = make_seqs(X_tr.index)
    Xs_ho, idx_ho = make_seqs(X_ho.index)
    y_seq = y_tr.loc[idx_tr].values
    w_seq = pd.Series(w_tr, index=X_tr.index).loc[idx_tr].values
    yho_seq, retho_seq = y_ho.loc[idx_ho].values, ret_ho.loc[idx_ho].values

    lstm = models.Sequential([
        layers.Input((seq_len, len(FEATURES))),
        layers.LSTM(32),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    lstm.compile(optimizer="adam", loss="binary_crossentropy")
    lstm.fit(Xs_tr, y_seq, sample_weight=w_seq,
             epochs=CONFIG["lstm_epochs"], batch_size=64, verbose=0)

    p = lstm.predict(Xs_ho, verbose=0).ravel()
    sig = (p >= 0.5).astype(int)
    r = sig * retho_seq
    sharpe = (r.mean() / (r.std() + 1e-9)) * np.sqrt(CONFIG["trading_days"]) if sig.sum() else np.nan
    rows.append({"Model": "LSTM (sequence)", "AUC": roc_auc_score(yho_seq, p),
                 "Accuracy": accuracy_score(yho_seq, sig),
                 "Strategy_Sharpe": sharpe, "Trades": int(sig.sum())})
except Exception as e:
    print("LSTM baseline skipped:", repr(e))

comparison = pd.DataFrame(rows).set_index("Model").round(4)
print("Baseline majority-class accuracy:", round(naive_acc, 4))
display(comparison)

# --- SHAP explainability for the tuned XGBoost ---
explainer = shap.TreeExplainer(xgb_best)
shap_values = explainer.shap_values(X_ho)
shap.summary_plot(shap_values, X_ho, show=True)

## 10. Event-Driven, Risk-Adjusted Backtest

Triple-barrier exits are event-driven (variable holding period), so we drive the
backtest off entry/exit times. To model a realistic single-position long-only
book we **greedily select non-overlapping trades** among the model's taken
signals, charge `cost_per_trade` per entry, build a daily equity curve, and
report Sharpe / Sortino / max-drawdown / Calmar plus a per-regime breakdown.

In [ ]:
cost = CONFIG["cost_per_trade"]
td = CONFIG["trading_days"]

# holdout event details
ho_events = events.loc[X_ho.index].copy()
ho_events["proba"] = proba_ho
ho_events["take"] = (ho_events["proba"] >= best_t)

# greedy non-overlapping selection among taken trades (single position at a time)
selected, last_exit = [], pd.Timestamp.min
for t0, row in ho_events.iterrows():
    if row["take"] and t0 > last_exit:
        selected.append(t0)
        last_exit = row["t1"]
trades = ho_events.loc[selected]

# daily strategy returns over the holdout window
ho_dates = featured.loc[X_ho.index[0]:X_ho.index[-1]].index
strat_daily = pd.Series(0.0, index=ho_dates)
for t0, row in trades.iterrows():
    span = featured.loc[t0:row["t1"]].index[1:]          # days held (after entry)
    if len(span) == 0:
        continue
    per_day = (1 + row["ret"]) ** (1 / len(span)) - 1
    strat_daily.loc[span] += per_day
    strat_daily.loc[t0] -= cost                          # entry cost

bench_daily = np.log(close.loc[ho_dates] / close.loc[ho_dates].shift(1)).fillna(0)
bench_daily = np.exp(bench_daily) - 1

def perf_stats(daily):
    eq = (1 + daily).cumprod()
    total = eq.iloc[-1] - 1
    ann = (1 + total) ** (td / len(daily)) - 1
    sharpe = daily.mean() / (daily.std() + 1e-12) * np.sqrt(td)
    downside = daily[daily < 0].std()
    sortino = daily.mean() / (downside + 1e-12) * np.sqrt(td)
    dd = (eq / eq.cummax() - 1).min()
    calmar = ann / abs(dd) if dd != 0 else np.nan
    return {"Total Return": total, "Annualized": ann, "Sharpe": sharpe,
            "Sortino": sortino, "Max Drawdown": dd, "Calmar": calmar}, eq

strat_stats, strat_eq = perf_stats(strat_daily)
bench_stats, bench_eq = perf_stats(bench_daily)

# --- equity & drawdown plots ---
fig, ax = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(bench_eq.index, bench_eq, color="gray", alpha=0.8, label="Buy & Hold Gold")
ax[0].plot(strat_eq.index, strat_eq, color="green", lw=2, label="Meta-Label Strategy")
ax[0].set_title("Out-of-Sample Equity Curve (net of costs)", fontweight="bold")
ax[0].set_ylabel("Growth of $1"); ax[0].legend(loc="upper left")
ax[1].fill_between(strat_eq.index, strat_eq / strat_eq.cummax() - 1, color="red", alpha=0.4)
ax[1].set_ylabel("Drawdown"); ax[1].set_xlabel("Date")
plt.tight_layout(); plt.show()

perf = pd.DataFrame({"Strategy": strat_stats, "Buy & Hold": bench_stats}).round(4)
display(perf)

# --- trade-level stats ---
wins = trades[trades["ret"] > 0]["ret"]
losses = trades[trades["ret"] <= 0]["ret"]
profit_factor = wins.sum() / abs(losses.sum()) if len(losses) and losses.sum() != 0 else np.inf
print(f"Trades taken:   {len(trades)}  (of {int(ho_events['take'].sum())} signals, "
      f"{len(ho_events)} candidate events)")
print(f"Hit rate:       {(trades['ret'] > 0).mean():.2%}")
print(f"Avg win / loss: {wins.mean():.4f} / {losses.mean():.4f}")
print(f"Profit factor:  {profit_factor:.2f}")
print(f"Avg holding:    {trades['holding'].mean():.1f} days")

# --- per-regime breakdown (regime measured at entry) ---
reg = featured.loc[trades.index]
trades_reg = trades.assign(
    Vol_Regime=np.where(reg["High_Vol_Regime"] == 1, "High Vol", "Low Vol"),
    Trend=np.where(reg["Trend_Sign"] == 1, "Uptrend", "Downtrend"))
breakdown = (trades_reg.groupby(["Vol_Regime", "Trend"])
             .agg(Trades=("ret", "size"), Avg_Return=("ret", "mean"),
                  Hit_Rate=("ret", lambda s: (s > 0).mean()))
             .round(4))
print("\nPer-regime breakdown:")
display(breakdown)